In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore

In [ ]:
df = pd.read_csv("Datasets\\raw\\Dataset.csv")

In [ ]:
# Output: (total Rows, total Features)
df.shape

In [ ]:
"""
Removing columns "Unnamed: 0" and "SepsisLabel" 
because "Unnamed: 0" is duplicate of the column "Hour" and 
"SepsisLabel" is target column not any feature
"""
columns_to_drop = ['Unnamed: 0', 'SepsisLabel']
df = df.drop(columns=columns_to_drop)
df.head()

In [ ]:
"""
Graphical analysis of whole data frame before pre-processing.

Heatmap of dataset before pre-processing"""
sns.heatmap(df.isnull(), cbar=False)

In [ ]:
"""BoxPlot analysis of all features together"""
plt.figure(figsize=(15,8))
df.boxplot(rot=90)
plt.show()

In [ ]:
"""BoxPlot analysis for each column"""
for col in df.columns:
    df.boxplot(column=col)
    plt.title(f"Boxplot - {col}")
    plt.show()

In [ ]:
"""Mathematical Analysis on whole data frame before pre-processing."""

In [ ]:
"""Skewness of all features, sorted in desending order."""
df.skew().sort_values(ascending=False)

In [ ]:
df.info()

In [ ]:
""" Nulls Percentage in each feature"""
df.isnull().mean()*100

In [ ]:
"""Feature wise analysis for decision making"""

In [ ]:
# skewness of HR = 0.43 almost normally distributed ..... this data is taken from earlier analysis.
df['HR'].describe()

In [ ]:
"""Histogram fo HR for distribution analysis"""
df['HR'].hist()
plt.show()

In [ ]:
"""
Physiological limit = 30-200
Values outside physiological limits are considered to be outliers,
and therefore will be removed.

For nulls:
Nulls will be replaced with patient-wise mean if possible, otherwise global mean.
"""
df[(df['HR']<=20) | (df['HR']>=300)][['Patient_ID', 'HR']]

In [ ]:
# skewness of O2Sat =  -4.15 Highly negative skewed ..... this data is taken from earlier analysis.
df['O2Sat'].describe()
"""
IQR = 3.5
UPPER_BOUND = 104.75
LOWER_BOUND = 90.75
"""

In [ ]:
"""Histogram fo O2Sat for distribution analysis"""
df['O2Sat'].hist()
plt.show()

In [ ]:
"""
Physiological limit = 50-100
Values outside physiological limits are considered to be outliers,
and therefore will be removed.

For nulls:
Nulls will be replaced with patient-wise median if possible, otherwise global median.
"""
df[(df['O2Sat']<50) | (df['O2Sat']>100)][['Patient_ID', 'O2Sat']]

In [ ]:
df['MAP'].describe()

In [ ]:
"""
Physiological limit = 30-200
Values outside physiological limits are considered to be outliers,
and therefore will be removed.

For nulls:
Nulls will be replaced with patient-wise median if possible, otherwise global median.
"""
df[(df['MAP']<30) | (df['MAP']>200)][['Patient_ID', 'MAP']]

In [ ]:
"""Histogram of 'MAP' for distribution analysis."""
df['MAP'].hist()
plt.show()

In [ ]:
"""
Relation between 'MAP', 'SBP', and 'DBP' is,

    SBP >= MAP >= DBP
    
Vlaues which doesn't satisfy the equation are outliers and will be removed.
"""
df[(df['MAP']<df['DBP']) | (df['MAP']>df['SBP'])][['MAP', 'DBP', 'SBP']]

In [ ]:
df['SBP'].describe()

In [ ]:
"""
Physiological limit = 40-250
Values outside physiological limits are considered to be outliers,
and therefore will be removed.

For nulls:
Nulls will be replaced with patient-wise median if possible, otherwise global median.
"""
df[(df['SBP']<40) | (df['SBP']>250)][['Patient_ID', 'SBP']]

In [ ]:
"""
Relation between 'MAP', 'SBP', and 'DBP' is,

    SBP >= MAP >= DBP
    
Vlaues which doesn't satisfy the equation are outliers and will be removed.
"""
df[(df['SBP']<df['DBP']) | (df['SBP']<df['MAP'])][['MAP', 'DBP', 'SBP']]

In [ ]:
df['Resp'].describe()

In [ ]:
"""
Physiological limit = 4-60
Values outside physiological limits are considered to be outliers,
and therefore will be removed.

For nulls:
Nulls will be replaced with patient-wise median if possible, otherwise global median.
"""
df[(df['Resp']<4) | (df['Resp']>60)][['Patient_ID', 'Resp']]

In [ ]:
"""
About Feature 'ICULOS': 

ICULOS is the length-of-stay (hours since ICU admit)

It is a time-series feature and therefore values far from mean/median does not mean the values are outliers.
ICULOS must strictly INCREASE overtime for a patient. 
Also value of ICULOS can't be negative.'

Therefore values not increasing with time for that patient, are possibly outliers and will be removed.
"""

df['ICULOS'].describe()

In [ ]:
df['ICULOS_diff'] = df.groupby('Patient_ID')['ICULOS'].diff()
df[df['ICULOS_diff'] < 0][['Patient_ID','ICULOS']]

In [ ]:
"""
About Feature 'HospAdmTime': 

HospAdmTime: Hours between hospital admit and ICU admit

It is a time-series feature and therefore values far from mean/median does not mean the values are outliers.
Outliers check condition: Each patient should have only 1 unique HospAdmTime, if not, the values are outliers and will be removed.
"""
df['HospAdmTime'].describe()

In [ ]:
"""Outliers detection condition implementation."""

df.groupby(by='Patient_ID')['HospAdmTime'].unique()

In [ ]:
"""Only 1 patient with ID=13777 has HospAdmTime null, as 1/43,336 is negligible, the patient will be removed."""
df[df['HospAdmTime'].isnull()][['HospAdmTime', 'Patient_ID']]

In [ ]:
"""
2nd Category of features begins here.
This category contains features with 20-40% missing values, and there analysis before pre-processing.
Features: 'Unit-1', 'Unit-2', and 'DBP'.

About Features:
Unit1 and Unit2 are binary and co-related features. If unit1 = 1 then for all such rows unit2 = 0, making one of them redundant.
Decision: Unit2 will be dropped as it is a derived feature(unit2 = unit1 - 1).

Similar checks of 'MAP' and 'SBP' are applied on 'DBP'
"""

In [ ]:
"""If unit1 = 1 but unit2 !=0 then such values are surely outliers and will be removed."""\

df[(df['Unit1']==1) & (df['Unit2']!=0)][['Patient_ID', 'Unit1', 'Unit2']]

In [ ]:
df['Unit1'].describe()

In [ ]:
df['DBP'].describe()

In [ ]:
"""
Physiological limit = 20-160
Values outside physiological limits are considered to be outliers,
and therefore will be removed.
Also values which doesn't satisfy the equation

    SBP >= MAP >= DBP 

will be removed.

For nulls:
Nulls will be replaced with patient-wise median if possible, otherwise global median.
"""
df[(df['DBP']>=df['MAP']) | (df['DBP']>=df['SBP']) | (df['DBP']<20) | (df['DBP']>160)][['Patient_ID', 'MAP', 'DBP', 'SBP']]